# Alpha Insight Engine: A Quantitative Corporate Financial Analysis Tool

## 1. Problem Definition & Target Audience
**Analytical Problem:** How can investors and financial analysts efficiently evaluate a company's financial health and earnings quality using raw accounting data? This project addresses the challenge of transforming complex datasets from WRDS into actionable visual insights to detect financial risks or potential for growth. I designed this tool to bridge the gap between raw database entries and high-level decision-making.

**Target Audience:** This tool is specifically developed for **Equity Research Analysts** and **Portfolio Managers** who require a rapid, interactive dashboard for preliminary due diligence on U.S. public companies.

## 2. Data Statement
* **Source:** WRDS (Wharton Research Data Services) - Compustat North America (Fundamental Quarterly).
* **Retrieval Date:** April 21, 2026.
* **Data Scope:** I extracted financial statement items from 2020 to 2026 for selected U.S. tickers to ensure a comprehensive view of post-pandemic corporate performance.



# I initialize the connection to the WRDS database here.
# This ensures that all subsequent analysis is based on authoritative corporate data.

In [ ]:
import wrds
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
from datetime import datetime


db = wrds.Connection() 

## Professional Financial Visualization Engine
I independently developed the `plot_financial_engine` function to visualize critical accounting dimensions:
* **Profitability:** I used ROA, ROE, and ROC to assess management's efficiency.
* **Liquidity & Solvency:** I monitored financial stability through Current/Quick ratios and liability trends.
* **Earnings Quality:** I compared Net Income with Operating Cash Flow to identify potential accounting distortions.

## Quantitative Risk Assessment
In this module, I focus on calculating risk-adjusted metrics. From an auditing perspective, evaluating **Annualized Volatility** and the **Sharpe Ratio** allows me to quantify how the market reacted to corporate disclosures and assess the quality of returns relative to the risk taken.

In [ ]:
def plot_financial_engine(df, idx):
    fig, axes = plt.subplots(6, 1, figsize=(12, 30)) 
    
    plot_configs = [
        (['roa', 'roe', 'roc'], 'Profitability Metrics (ROA/ROE/ROC)', 0),
        (['current_ratio', 'quick_ratio'], 'Liquidity Analysis', 1),
        (['roa_ma'], 'ROA 4-Quarter Moving Average (Trend)', 2),
        (['ret'], 'Quarterly Returns (%)', 3),
        
        (['niq', 'oancfy'], 'Earnings Quality: Net Income vs. Operating Cash Flow', 4),
        (['ltq'], 'Total Liability Trend (Solvency Perspective)', 5)
    ]
    
    for cols, title, i in plot_configs:
        for col in cols:
            if i == 4:
                
                axes[i].bar(df['datadate'], df[col], label=col.upper(), width=70, alpha=0.6)
            else:
                axes[i].plot(df['datadate'], df[col], label=col.upper(), marker='o', markersize=5, linewidth=2)
            
            
            axes[i].scatter(df.loc[idx, 'datadate'], df.loc[idx, col], color='red', s=150, edgecolors='black', zorder=10)
        
        axes[i].set_title(title, fontsize=14, fontweight='bold')
        axes[i].legend()
        axes[i].grid(True, linestyle=':', alpha=0.6)
        
    plt.tight_layout()
    plt.show()


def calculate_risk_metrics(df, idx):
    recent_rets = df.loc[idx:, 'ret'].dropna()
    if len(recent_rets) < 2:
        print("Note: Insufficient data points after the selected date to calculate risk metrics.")
       
        return
        
    
    vol = recent_rets.std() * np.sqrt(4)
    sharpe = (recent_rets.mean() / recent_rets.std()) * np.sqrt(4) if recent_rets.std() != 0 else 0
    
    
   

  
    print(f"\n--- 📊 Risk & Performance Metrics (From Selected Date) ---")
    print(f"📈 Annualized Volatility: {vol:.2%}")
    print(f"⚖️ Risk-Adjusted Return (Sharpe Ratio): {sharpe:.2f}")
    print("-" * 50)


## Interactive Analysis & Deep Value Reporting
This is the core execution layer of my project, integrating the UI with business logic:
1. **SQL Automation:** I wrote efficient SQL to extract raw accounts and calculate ratios server-side.
2. **Automated Health Checks:** I implemented triggers for liquidity warnings based on the Current Ratio.
3. **Deep Accounting Insight:** I concluded the report by evaluating **Earnings Quality** via the Accrual Ratio and assessing **Capital Structure** to provide a qualitative judgment on the firm's overall financial health.

In [ ]:
ticker_input = widgets.Text(value='NVDA', description='US Ticker:', placeholder='For Example: TSLA')
date_input = widgets.DatePicker(
    description='Select Date:',
    value=datetime(2024, 4, 8).date(),
    min=datetime(2021, 1, 1).date(),
    max=datetime(2026, 1, 1).date()
)
btn = widgets.Button(description="Generate Deep Value Report", button_style='danger')
out = widgets.Output()


        

def on_click_analyze(b):
    with out:
        clear_output(wait=True)
        ticker = ticker_input.value.strip().upper()
        sel_date = pd.to_datetime(date_input.value).normalize()

       
        

        query = f"""
       SELECT datadate, 
       prccq,
       niq, 
       oancfy, 
       ltq, 
       atq,
       ceqq,
       (niq / NULLIF(atq, 0)) as roa,
       (niq / NULLIF(ceqq, 0)) as roe,
       (oiadpq / NULLIF(atq - lctq, 0)) as roc,
       (actq / NULLIF(lctq, 0)) as current_ratio,
       ((cheq + rectq) / NULLIF(lctq, 0)) as quick_ratio
FROM comp.fundq
WHERE tic = '{ticker}'
  AND datadate >= '2020-01-01'
  AND datadate <= '2026-01-01'
  AND indfmt = 'INDL'
  AND datafmt = 'STD'
  AND popsrc = 'D'
  AND consol = 'C'
ORDER BY datadate
"""




        try:
            df = db.raw_sql(query)
            if df is None or df.empty:
                print(f"❌ No financial data found for {ticker}.")
                return

            df['datadate'] = pd.to_datetime(df['datadate'])
            df['roa_ma'] = df['roa'].rolling(window=4).mean()
            df['ret'] = df['prccq'].pct_change()

            idx = (df['datadate'] - sel_date).abs().idxmin()
            latest = df.loc[idx]

            print(f"🚀 Alpha Insight Engine | Report Target: {ticker}")
            print(f"📅 Selected Quarter: {latest['datadate'].date()}")
            print("-" * 40)

            if latest['current_ratio'] < 1.2:
                print("🔴 Solvency Alert: Current Ratio is below 1.2. Please monitor short-term cash flow.")
            else:
                print("🟢 Financial Health: Strong liquidity with solid short-term solvency.")

            post_data = df.loc[idx:]
            if len(post_data) > 1:
                total_gain = (post_data['prccq'].iloc[-1] / latest['prccq'] - 1) * 100
                print(f"💰 Value Simulation: Total return since entry at this point is {total_gain:.2f}%")

            plot_financial_engine(df, idx)
            calculate_risk_metrics(df, idx)
               
            print(f"\n--- 📈 Deep Accounting & Financial Health Insight ---")
    
   
            accrual_ratio = (latest['niq'] - latest['oancfy']) / abs(latest['niq']) if latest['niq'] != 0 else 0
            print(f"💎 Earnings Quality: {'✅ High (Cash-Backed)' if accrual_ratio < 0.1 else '⚠️ Low (Accrual-Heavy)'}")
            print(f"   (Accrual Ratio: {accrual_ratio:.2%})")

   
            debt_ratio = latest['ltq'] / latest['atq'] if latest['atq'] != 0 else 0
            print(f"🛡️ Financial Health Value: {'🟢 Optimal' if debt_ratio < 0.5 else '🚩 High Leverage'}")
            print(f"   (Total Debt-to-Assets: {debt_ratio:.2%})")
    
            print("-" * 50)


        except Exception as e:
            print(f"⚠️ Error occurred during analysis: {e}")

btn.on_click(on_click_analyze)
display(widgets.VBox([ticker_input, date_input, btn]), out)


## Resource Management
To ensure professional practice and system efficiency, I close the database connection in this final step to release all allocated resources.

In [ ]:
db.close()